# Notebook 11: ResNet

**Module:** 06 CNN  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Understand skip/residual connections
2. Implement ResNet block
3. Connect to water-bodies SE-ResNet50 encoder
4. Train ResNet on CIFAR-10

## ResNet (He et al., 2016)

**Problem:** Very deep networks degrade (vanishing gradients).

**Solution:** Skip connection: $y = F(x) + x$

Learn residual $F(x)$ instead of desired mapping $H(x)$.

**Your water-bodies-detection** uses **SE-ResNet50** encoder — ResNet50 + Squeeze-and-Excitation channel attention.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
transform = T.Compose([T.ToTensor(), T.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)
classes = train_set.classes
print(f'CIFAR-10: {len(train_set)} train, {len(test_set)} test, {len(classes)} classes')

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total, correct, loss_sum = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        total += len(y)
    return loss_sum/total, correct/total

def evaluate(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            loss = criterion(out, y)
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return loss_sum/total, correct/total

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm2d(out_ch))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

class ResNetSmall(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3,64,3,padding=1,bias=False), nn.BatchNorm2d(64), nn.ReLU())
        self.layer1 = ResidualBlock(64, 64)
        self.layer2 = ResidualBlock(64, 128, stride=2)
        self.layer3 = ResidualBlock(128, 256, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.stem(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        return self.fc(self.pool(x).flatten(1))

model = ResNetSmall()
print(f'ResNet-Small params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Pretrained ResNet50 — same family as water-bodies encoder
from torchvision.models import resnet50, ResNet50_Weights
pretrained = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
print('ResNet50 layer names (first 10):', [n for n,_ in list(pretrained.named_children())[:10]])
print('\nwater-bodies uses se_resnet50 — ResNet50 + Squeeze-Excitation')

## Exercise

Train ResNetSmall for 10 epochs on full CIFAR-10. Target: >70% test accuracy.

In [ ]:
# YOUR CODE HERE


---

## Summary

ResNet skip connections enable training of very deep networks. Your segmentation encoder is ResNet-based.

**Next:** [12_DenseNet.ipynb](12_DenseNet.ipynb)